###  load data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [3]:
# Go up one level from Notebook folder to the project root
os.chdir(r'C:\Users\workstation\Desktop\customer_churn_prediction')

# Verify
print(os.getcwd())

C:\Users\workstation\Desktop\customer_churn_prediction


In [3]:
# Load helper
def load_logs(filepath):
    chunks = []
    for chunk in pd.read_csv(filepath, chunksize=1_000_000):
        chunk['date'] = pd.to_datetime(chunk['date'], format='%Y%m%d')
        chunks.append(chunk)
    return pd.concat(chunks)


In [4]:
# 1. User Logs 
logs = pd.concat([
    load_logs(r'data\raw\User_logs.csv'),
    load_logs(r'data\raw\User_logs_v2.csv')
], ignore_index=True)

In [4]:
# 2. Transactions 
trans = pd.concat([
    pd.read_csv(r'data\raw\transactions_v2.csv'),
    pd.read_csv(r'data\raw\transactions.csv')
], ignore_index=True)

In [5]:
#3. members
members = pd.read_csv(r'data\raw\members_v3.csv')

In [6]:
#4. train
train = pd.concat([
    pd.read_csv(r'data\raw\train_v2.csv'),
    pd.read_csv(r'data\raw\train.csv')
], ignore_index=True)

**data cleaning**

In [ ]:

both_zero = ((trans['plan_list_price'] == 0) & (trans['actual_amount_paid'] == 0)).sum()
print(f"Both price and amount = 0  : {both_zero:,}")

price_zero_only = ((trans['plan_list_price'] == 0) & (trans['actual_amount_paid'] > 0)).sum()
print(f"Price = 0 but amount > 0   : {price_zero_only:,}")


amount_zero_only = ((trans['actual_amount_paid'] == 0) & (trans['plan_list_price'] > 0)).sum()
print(f"Amount = 0 but price > 0   : {amount_zero_only:,}")

print("\nPlan days when price = 0:")
print(trans[trans['plan_list_price'] == 0]['payment_plan_days'].value_counts().head(10))

Both price and amount = 0  : 657,473
Price = 0 but amount > 0   : 859,484
Amount = 0 but price > 0   : 560,851

Plan days when price = 0:
Series([], Name: count, dtype: int64)


In [10]:
# TRANSACTIONS 

print("Step 1: Running Data Cleaning on Transactions...")

# 1. Create a working copy of your loaded transactions dataframe
trans_cleaned = trans.copy()

# 2. Immediately drop the duplicate rows discovered in your analysis
trans_cleaned.drop_duplicates(inplace=True)

# 3. Convert date columns from raw int64 format to proper DateTime format
trans_cleaned["transaction_date"] = pd.to_datetime(
    trans_cleaned["transaction_date"], format="%Y%m%d", errors="coerce"
)
trans_cleaned["membership_expire_date"] = pd.to_datetime(
    trans_cleaned["membership_expire_date"], format="%Y%m%d", errors="coerce"
)

# 4. Handle invalid/suspicious expiration dates using your EDA bounds (2015 to 2020)
valid_expire_mask = (trans_cleaned["membership_expire_date"].dt.year >= 2015) & (
    trans_cleaned["membership_expire_date"].dt.year <= 2020
)
trans_cleaned.loc[~valid_expire_mask, "membership_expire_date"] = np.nan

# 5. Impute any resulting NaNs with the respective column median dates
median_expire_date = trans_cleaned["membership_expire_date"].median()
trans_cleaned["membership_expire_date"] = trans_cleaned[
    "membership_expire_date"
].fillna(median_expire_date)

median_trans_date = trans_cleaned["transaction_date"].median()
trans_cleaned["transaction_date"] = trans_cleaned["transaction_date"].fillna(
    median_trans_date
)

print("Data Cleaning Complete! Checking for any remaining missing values:")
print(trans_cleaned.isnull().sum())
print(f"Cleaned Transactions Shape: {trans_cleaned.shape}")


Step 1: Running Data Cleaning on Transactions...
Data Cleaning Complete! Checking for any remaining missing values:
msno                      0
payment_method_id         0
payment_plan_days         0
plan_list_price           0
actual_amount_paid        0
is_auto_renew             0
transaction_date          0
membership_expire_date    0
is_cancel                 0
dtype: int64
Cleaned Transactions Shape: (22975416, 9)


In [ ]:

# MEMBERS 

print("Step 1: Running Data Cleaning on Members...")

# 1. Use your already loaded 'members' dataframe to safely apply changes
members_cleaned = members.copy()

# 2. Clean age column (bd): keep valid ages between 10 and 80, set others to NaN
members_cleaned["bd"] = members_cleaned["bd"].apply(
    lambda x: x if (10 <= x <= 80) else np.nan
)

# 3. Impute missing ages using the median of each city
members_cleaned["bd"] = members_cleaned.groupby("city")["bd"].transform(
    lambda x: x.fillna(x.median())
)
# Fill any remaining NaNs with the global median
members_cleaned["bd"] = members_cleaned["bd"].fillna(members_cleaned["bd"].median())

# 4. Clean gender column: fill missing values (65%) with 'unknown' category
members_cleaned["gender"] = members_cleaned["gender"].fillna("unknown")

# 5. Clean dates: convert from int64 (e.g., 20151015) to proper DateTime format
members_cleaned["registration_init_time"] = pd.to_datetime(
    members_cleaned["registration_init_time"], format="%Y%m%d", errors="coerce"
)
# Impute invalid or missing dates with the most frequent date (Mode)
members_cleaned["registration_init_time"] = members_cleaned[
    "registration_init_time"
].fillna(members_cleaned["registration_init_time"].mode()[0])

print("Data Cleaning Complete! Checking if any nulls remain:")
print(members_cleaned.isnull().sum())

Step 1: Running Data Cleaning on Members...
Data Cleaning Complete! Checking if any nulls remain:
msno                      0
city                      0
bd                        0
gender                    0
registered_via            0
registration_init_time    0
dtype: int64


In [ ]:
#user_logs
import pyarrow as pa
import pyarrow.parquet as pq

# 1. Define upper bounds (99th Percentile)
limits = {
    "num_25": 61.0,
    "num_50": 16.0,
    "num_75": 8.0,
    "num_985": 9.0,
    "num_100": 182.0,
    "num_unq": 156.0,
    "total_secs": 44683.2,
}

# Input and output file paths
input_files = [r"data\raw\User_logs.csv", r"data\raw\User_logs_v2.csv"]
output_file = r"data\processed\user_logs_cleaned.parquet"

# Ensure processed directory exists
os.makedirs(os.path.dirname(output_file), exist_ok=True)

chunk_size = 10_000_000
parquet_writer = None

# Defining explicit schema for PyArrow to prevent type mismatch across chunks
# We force 'msno' to be string and 'date' to be timestamp
arrow_schema = pa.schema([
    ('msno', pa.string()),
    ('date', pa.timestamp('ns')),
    ('num_25', pa.int8()),
    ('num_50', pa.int8()),
    ('num_75', pa.int8()),
    ('num_985', pa.int8()),
    ('num_100', pa.int16()),
    ('num_unq', pa.int16()),
    ('total_secs', pa.float32())
])

print("Starting streaming, cleaning, and optimization with enforced schema...")

try:
    for file_path in input_files:
        print(f"Processing file: {file_path}")
        
        # Enforce 'msno' as string directly during CSV reading to avoid null type inference
        for chunk in pd.read_csv(file_path, chunksize=chunk_size, parse_dates=["date"], dtype={"msno": str}):
            
            # A. Filter out impossible values
            chunk = chunk[(chunk["total_secs"] > 0) & (chunk["total_secs"] <= 86400)]
            
            # B. Apply Winsorization (Capping outliers)
            for col, limit in limits.items():
                chunk[col] = np.clip(chunk[col], a_min=None, a_max=limit)
                
            # C. Downcasting to match our strict Arrow Schema
            chunk["num_25"] = chunk["num_25"].astype(np.int8)
            chunk["num_50"] = chunk["num_50"].astype(np.int8)
            chunk["num_75"] = chunk["num_75"].astype(np.int8)
            chunk["num_985"] = chunk["num_985"].astype(np.int8)
            chunk["num_100"] = chunk["num_100"].astype(np.int16)
            chunk["num_unq"] = chunk["num_unq"].astype(np.int16)
            chunk["total_secs"] = chunk["total_secs"].astype(np.float32)
            
            # D. Convert Pandas DataFrame to PyArrow Table using the enforced strict schema
            table = pa.Table.from_pandas(chunk, schema=arrow_schema, preserve_index=False)
            
            # E. Initialize the ParquetWriter with the fixed schema
            if parquet_writer is None:
                parquet_writer = pq.ParquetWriter(output_file, arrow_schema, compression='snappy')
                
            # F. Write the current table/chunk directly to disk
            parquet_writer.write_table(table)
            
finally:
    # Always close the writer to ensure the file is correctly saved
    if parquet_writer is not None:
        parquet_writer.close()

print(f"Success! Cleaned and compressed file saved at: {output_file}")

Starting streaming, cleaning, and optimization with enforced schema...
Processing file: data\raw\User_logs.csv
Processing file: data\raw\User_logs_v2.csv
Success! Cleaned and compressed file saved at: data\processed\user_logs_cleaned.parquet


In [12]:
#train

print("Checking Train dataset...")

# Load your train file (Update the path and filename 'train.csv' or 'train_v2.csv' if needed)
train_path = r"data\raw\train.csv"
train = pd.read_csv(train_path)

# 1. Check for missing values or duplicates
print(f"Train Shape: {train.shape}")
print(f"Missing values in Train:\n{train.isnull().sum()}")
print(f"Duplicate rows in Train: {train.duplicated().sum()}")

# 2. Drop duplicates just in case
train.drop_duplicates(inplace=True)

# 3. Check Churn distribution (Class Imbalance)
print("\nTarget Distribution (is_churn):")
print(train["is_churn"].value_counts())
print("\nTarget Percentage:")
print(train["is_churn"].value_counts(normalize=True) * 100)

Checking Train dataset...
Train Shape: (992931, 2)
Missing values in Train:
msno        0
is_churn    0
dtype: int64
Duplicate rows in Train: 0

Target Distribution (is_churn):
is_churn
0    929460
1     63471
Name: count, dtype: int64

Target Percentage:
is_churn
0    93.607713
1     6.392287
Name: proportion, dtype: float64


---
### Construct Data (Feature Engineering)

In [ ]:
#user_logs
import duckdb

# Base project directory setup dynamically
project_dir = r"C:\Users\workstation\Desktop\customer_churn_prediction"
cleaned_parquet = os.path.join(project_dir, r"data\processed\user_logs_cleaned.parquet")
output_features_file = os.path.join(project_dir, r"data\processed\user_logs_features.parquet")

# Force using forward slashes for DuckDB paths to prevent escape-character bugs
cleaned_parquet_sql = cleaned_parquet.replace("\\", "/")
output_features_file_sql = output_features_file.replace("\\", "/")

print("Starting ultra-fast On-Disk Aggregation using DuckDB...")
print(f"Reading from: {cleaned_parquet_sql}")

con = duckdb.connect(database=':memory:')

query = f"""
    SELECT 
        msno,
        SUM(num_100) AS num_100,
        (SUM(num_unq)::DOUBLE / (COUNT(date) + 1e-5)) AS num_unq,
        SUM(total_secs) AS total_secs,
        MAX(date) AS latest_date,
        COUNT(date) AS total_active_days,
        (SUM(num_25 + num_50)::DOUBLE / (SUM(num_25 + num_50 + num_75 + num_985 + num_100) + 1e-5)) AS skip_ratio,
        (SUM(num_100)::DOUBLE / ((SUM(num_unq)::DOUBLE / (COUNT(date) + 1e-5)) + 1e-5)) AS loop_ratio,
        (SUM(total_secs)::DOUBLE / (COUNT(date) + 1e-5)) AS secs_per_active_day
    FROM parquet_scan('{cleaned_parquet_sql}')
    GROUP BY msno
"""

print("Executing SQL query and streaming results straight to a new Parquet file...")
con.execute(f"COPY ({query}) TO '{output_features_file_sql}' (FORMAT PARQUET);")

print(f"Success! Master features generated seamlessly via DuckDB. Saved at: {output_features_file}")

Starting ultra-fast On-Disk Aggregation using DuckDB...
Reading from: C:/Users/workstation/Desktop/customer_churn_prediction/data/processed/user_logs_cleaned.parquet
Executing SQL query and streaming results straight to a new Parquet file...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! Master features generated seamlessly via DuckDB. Saved at: C:\Users\workstation\Desktop\customer_churn_prediction\data\processed\user_logs_features.parquet


In [7]:


project_dir = r"C:\Users\workstation\Desktop\customer_churn_prediction"
output_features_file = os.path.join(project_dir, r"data\processed\user_logs_features.parquet")

# 1. Read just the first 10 rows to inspect the structure
print("--- Sample Rows from the Feature File ---")
df_sample = pd.read_parquet(output_features_file, engine="pyarrow").head(10)
display(df_sample)

# 2. Print information about columns, missing values, and memory
print("\n--- File Info & Summary Statistics ---")
df_info = pd.read_parquet(output_features_file, engine="pyarrow")
print(df_info.info())
print(f"\nTotal unique users extracted: {len(df_info):,}")

--- Sample Rows from the Feature File ---


,msno,num_100,num_unq,total_secs,latest_date,total_active_days,skip_ratio,loop_ratio,secs_per_active_day
0,orbceYkC20hZ6qSegPwyJOxK3gyCsnayFgHduLhjrW0=,2278.0,17.213792,7.360606e+05,2017-03-25,145,0.161321,132.335669,5076.279387
1,AQif61CzscuFuLb0DkSs8XxFZUS/23BVO8M3yvSHY+0=,674.0,9.618420,2.396789e+05,2015-05-15,76,0.342264,70.073808,3153.669707
2,rErWAoDOb/omTsY1VpxlYr+oguwQtp2sx+QCd82Qu7o=,23419.0,44.680341,6.735208e+06,2017-03-31,585,0.172649,524.145389,11513.175062
3,mlJDEAwXabuIPapJPix3cOJuFaNr3xqua5a1AUSTJG8=,57.0,4.583330,1.709960e+04,2016-09-22,12,0.109756,12.436347,1424.965155
4,/YvUuYQTTKSKK4UB6bnTprI56b3WHj8HuAhcuqhuNbs=,8.0,6.666644,4.094012e+03,2015-03-23,3,0.555555,1.200002,1364.666149
5,DmEltw+0DqNuKMDcWOh+7FqkEdaPDOpGXnIrwMTla1s=,8831.0,18.350909,2.907558e+06,2017-03-31,550,0.329986,481.229312,5286.469473
6,9ZZQt+cJJ7JPUZdgpHNAsP9pC5b4iLoJusMyGsrnZjo=,481.0,11.136361,1.841190e+05,2016-05-17,44,0.334146,43.191808,4184.522614
7,S2mLxjakLdg3K6MliS+W3LEujqLQIo3zUmlHyWp5whM=,8.0,5.249987,3.291893e+03,2015-09-13,4,0.800000,1.523810,822.971202
8,WXcATIDHLXYeQOfxKLl49+BZcYI1n/LiKeXrlKrGqUI=,7026.0,29.110700,2.082398e+06,2017-03-30,271,0.264411,241.354470,7684.126591
9,O4Jh6RVmGWR6bdt1qQLaIT8Eli729weEqzQGIn9KmAc=,2996.0,13.761363,9.930082e+05,2017-03-30,264,0.366256,217.710833,3761.394490



--- File Info & Summary Statistics ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5298323 entries, 0 to 5298322
Data columns (total 9 columns):
 #   Column               Dtype         
---  ------               -----         
 0   msno                 object        
 1   num_100              float64       
 2   num_unq              float64       
 3   total_secs           float64       
 4   latest_date          datetime64[ns]
 5   total_active_days    int64         
 6   skip_ratio           float64       
 7   loop_ratio           float64       
 8   secs_per_active_day  float64       
dtypes: datetime64[ns](1), float64(6), int64(1), object(1)
memory usage: 363.8+ MB
None

Total unique users extracted: 5,298,323


In [9]:
#members


project_dir = r"C:\Users\workstation\Desktop\customer_churn_prediction"
output_members_file = os.path.join(
    project_dir, r"data\processed\members_cleaned.parquet"
)

print("Step 2: Running Feature Engineering on Cleaned Members...")

# 1. Extract numerical components from the registration date
members_features = members_cleaned.copy()
members_features["reg_year"] = members_features["registration_init_time"].dt.year
members_features["reg_month"] = members_features["registration_init_time"].dt.month
members_features["reg_day"] = members_features["registration_init_time"].dt.day

# 2. Calculate account age in days up to the snapshot date (End of April 2017)
snapshot_date = pd.to_datetime("2017-04-30")
members_features["account_age_days"] = (
    snapshot_date - members_features["registration_init_time"]
).dt.days

# 3. Drop the original raw date column as tree models cannot process it directly
members_features.drop(columns=["registration_init_time"], inplace=True)

# 4. Save the final processed dataset as Parquet for memory efficiency and fast merging
members_features.to_parquet(output_members_file, engine="pyarrow", index=False)

print(f"Success! Master Features saved as Parquet at: {output_members_file}")
print(f"Final Table Shape: {members_features.shape}")
display(members_features.head())

Step 2: Running Feature Engineering on Cleaned Members...
Success! Master Features saved as Parquet at: C:\Users\workstation\Desktop\customer_churn_prediction\data\processed\members_cleaned.parquet
Final Table Shape: (6769473, 9)


,msno,city,bd,gender,registered_via,reg_year,reg_month,reg_day,account_age_days
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,27.0,unknown,11,2011,9,11,2058
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,27.0,unknown,7,2011,9,14,2055
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,27.0,unknown,11,2011,9,15,2054
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,27.0,unknown,11,2011,9,15,2054
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32.0,female,9,2011,9,15,2054


In [11]:
#trans


project_dir = r"C:\Users\workstation\Desktop\customer_churn_prediction"
output_transactions_file = os.path.join(
    project_dir, r"data\processed\transactions_cleaned.parquet"
)

print("Step 2: Aggregating Transactions to User-Level Features...")

# 1. Group by user (msno) and extract elite historical behavior features
trans_features = (
    trans_cleaned.groupby("msno")
    .agg(
        total_transactions=("transaction_date", "count"),
        total_actual_paid=("actual_amount_paid", "sum"),
        mean_actual_paid=("actual_amount_paid", "mean"),
        mean_plan_days=("payment_plan_days", "mean"),
        total_cancellations=("is_cancel", "sum"),
        latest_auto_renew=("is_auto_renew", "last"),
        latest_payment_method=("payment_method_id", "last"),
        latest_transaction_date=("transaction_date", "max"),
        latest_membership_expire_date=("membership_expire_date", "max"),
    )
    .reset_index()
)

# 2. Advanced Engineered Feature: Average price variance (overpayment/underpayment delta)
mean_paid = trans_cleaned.groupby("msno")["actual_amount_paid"].mean().values
mean_price = trans_cleaned.groupby("msno")["plan_list_price"].mean().values
trans_features["average_under_overpayment"] = mean_paid - mean_price

# 3. Save the master aggregated file as Parquet for optimal downstream merging
trans_features.to_parquet(
    output_transactions_file, engine="pyarrow", index=False
)

print(
    f"Success! Master Transaction Features saved as Parquet at: {output_transactions_file}"
)
print(f"Final Aggregated Table Shape: {trans_features.shape}")
display(trans_features.head())

Step 2: Aggregating Transactions to User-Level Features...
Success! Master Transaction Features saved as Parquet at: C:\Users\workstation\Desktop\customer_churn_prediction\data\processed\transactions_cleaned.parquet
Final Aggregated Table Shape: (2426143, 11)


,msno,total_transactions,total_actual_paid,mean_actual_paid,mean_plan_days,total_cancellations,latest_auto_renew,latest_payment_method,latest_transaction_date,latest_membership_expire_date,average_under_overpayment
0,+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=,1,0,0.0,7.000000,0,0,35,2016-09-09,2016-09-14,0.000000
1,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,2,3387,1693.5,402.500000,0,0,38,2016-10-23,2018-02-06,0.000000
2,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,5,495,99.0,30.000000,0,1,41,2017-03-15,2017-04-15,0.000000
3,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,21,3129,149.0,28.714286,0,1,39,2017-03-31,2017-05-19,7.095238
4,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,27,4023,149.0,28.888889,0,1,41,2017-03-26,2017-04-26,5.518519


### merge data

In [17]:


# Define paths to your processed Parquet files
project_dir = r"C:\Users\workstation\Desktop\customer_churn_prediction"
processed_dir = os.path.join(project_dir, r"data\processed")

# 1. Start with the core Train target dataset (Assuming 'train' dataframe is already in RAM)
# If it's not loaded, uncomment the line below:
# train = pd.read_csv(os.path.join(project_dir, r"data\raw\train.csv"))
integrated_df = train.copy()
print(f"Initial Train dataset shape: {integrated_df.shape}")

# 2. Merge Transactions Features
print("Merging Transactions features...")
trans_df = pd.read_parquet(
    os.path.join(processed_dir, "transactions_cleaned.parquet")
)
integrated_df = pd.merge(integrated_df, trans_df, on="msno", how="left")
del trans_df  # Free memory immediately

# 3. Merge Members Features
print("Merging Members features...")
members_df = pd.read_parquet(
    os.path.join(processed_dir, "members_cleaned.parquet")
)
integrated_df = pd.merge(integrated_df, members_df, on="msno", how="left")
del members_df

# 4. Merge User Logs Features
print("Merging User Logs features...")
logs_df = pd.read_parquet(
    os.path.join(processed_dir, "user_logs_features.parquet")
)
integrated_df = pd.merge(integrated_df, logs_df, on="msno", how="left")
del logs_df

print("\nTask 4 Complete! All data sources unified successfully.")
print(f"Unified Dataset Shape: {integrated_df.shape}")
display(integrated_df.head())

Initial Train dataset shape: (992931, 2)
Merging Transactions features...
Merging Members features...
Merging User Logs features...

Task 4 Complete! All data sources unified successfully.
Unified Dataset Shape: (992931, 28)


,msno,is_churn,total_transactions,total_actual_paid,mean_actual_paid,mean_plan_days,total_cancellations,latest_auto_renew,latest_payment_method,latest_transaction_date,...,reg_day,account_age_days,num_100,num_unq,total_secs,latest_date,total_active_days,skip_ratio,loop_ratio,secs_per_active_day
0,waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,1,2,149,74.500000,18.500000,0,0,38,2017-01-07,...,6.0,4407.0,409.0,17.769224,1.514166e+05,2017-02-08,26.0,0.127863,23.017312,5823.713447
1,QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,1,23,3458,150.347826,28.826087,2,1,39,2017-02-24,...,7.0,4406.0,9379.0,14.001953,3.036231e+06,2017-03-19,512.0,0.151742,669.834658,5930.139491
2,fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,1,10,1492,149.200000,30.000000,1,0,36,2017-01-12,...,16.0,4214.0,9814.0,48.502108,2.951128e+06,2017-01-31,237.0,0.187306,202.341681,12452.014450
3,mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,1,3,1937,645.666667,283.333333,0,0,17,2017-03-27,...,2.0,4197.0,17344.0,25.222069,5.987051e+06,2017-03-31,725.0,0.324119,687.651492,8258.001404
4,XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,1,9,4053,450.333333,97.222222,0,0,38,2017-02-25,...,28.0,4141.0,64979.0,92.715788,1.850374e+07,2017-03-31,760.0,0.053332,700.840647,24347.028147


### format data

In [18]:


# 1. Data Type Conversions: Convert raw dates into numeric day deltas
print("Converting raw date columns into numeric day deltas...")
snapshot_date = pd.to_datetime("2017-04-30")

# Calculate Days Since Last Transaction
integrated_df["days_since_last_transaction"] = (
    snapshot_date - integrated_df["latest_transaction_date"]
).dt.days

# Calculate Days Until Membership Expiration (Negative values mean expired)
integrated_df["days_until_expiration"] = (
    integrated_df["latest_membership_expire_date"] - snapshot_date
).dt.days

# Drop the raw datetime objects as they cannot be fed to tree models
integrated_df.drop(
    columns=["latest_transaction_date", "latest_membership_expire_date"],
    inplace=True,
)

# 2. Impute any new missing values introduced by the Left Joins safely
print("Handling missing values created during merge...")
for col in integrated_df.select_dtypes(include=["number"]).columns:
    if col != "is_churn":
        integrated_df[col] = integrated_df[col].fillna(
            integrated_df[col].median()
        )

# Fix gender missing values
integrated_df["gender"] = integrated_df["gender"].fillna("unknown")

# 3. Column Reordering: Move the Target column (is_churn) to the very end
cols = [col for col in integrated_df.columns if col != "is_churn"] + [
    "is_churn"
]
final_modeling_df = integrated_df[cols]

# 4. Saving the Prepared Dataset as CSV
final_csv_path = os.path.join(processed_dir, "final_modeling_dataset.csv")
print(f"Exporting final clean dataset to CSV at: {final_csv_path}")
final_modeling_df.to_csv(final_csv_path, index=False)

print("\nTASK 5 COMPLETE & PHASE SUCCESSFUL! ")
print(f"Final Structured Shape for Modeling: {final_modeling_df.shape}")
display(final_modeling_df.head())

Converting raw date columns into numeric day deltas...
Handling missing values created during merge...
Exporting final clean dataset to CSV at: C:\Users\workstation\Desktop\customer_churn_prediction\data\processed\final_modeling_dataset.csv

TASK 5 COMPLETE & PHASE SUCCESSFUL! 
Final Structured Shape for Modeling: (992931, 28)


,msno,total_transactions,total_actual_paid,mean_actual_paid,mean_plan_days,total_cancellations,latest_auto_renew,latest_payment_method,average_under_overpayment,city,...,num_unq,total_secs,latest_date,total_active_days,skip_ratio,loop_ratio,secs_per_active_day,days_since_last_transaction,days_until_expiration,is_churn
0,waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,2,149,74.500000,18.500000,0,0,38,0.000000,18.0,...,17.769224,1.514166e+05,2017-02-08,26.0,0.127863,23.017312,5823.713447,113,-83,1
1,QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,23,3458,150.347826,28.826087,2,1,39,6.478261,10.0,...,14.001953,3.036231e+06,2017-03-19,512.0,0.151742,669.834658,5930.139491,65,-40,1
2,fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,10,1492,149.200000,30.000000,1,0,36,0.000000,11.0,...,48.502108,2.951128e+06,2017-01-31,237.0,0.187306,202.341681,12452.014450,108,-86,1
3,mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,3,1937,645.666667,283.333333,0,0,17,0.000000,13.0,...,25.222069,5.987051e+06,2017-03-31,725.0,0.324119,687.651492,8258.001404,34,-4,1
4,XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,9,4053,450.333333,97.222222,0,0,38,0.000000,3.0,...,92.715788,1.850374e+07,2017-03-31,760.0,0.053332,700.840647,24347.028147,64,28,1


In [20]:


# 1. Impute any missing values created during merge safely
print("Handling missing values...\n")
for col in integrated_df.select_dtypes(include=["number"]).columns:
    if col != "is_churn":
        integrated_df[col] = integrated_df[col].fillna(
            integrated_df[col].median()
        )

# Fix gender missing values before encoding (if gender exists in data)
if "gender" in integrated_df.columns:
    integrated_df["gender"] = integrated_df["gender"].fillna("unknown")

    # 2. ADVANCED FORMATTING: Convert Text (Gender) to Numeric (0 and 1)
    print("Encoding categorical column 'gender' into numeric flags...\n")
    integrated_df = pd.get_dummies(
        integrated_df, columns=["gender"], drop_first=True, dtype=int
    )


# 3. Column Reordering: Move the Target column (is_churn) to the very end
cols = [col for col in integrated_df.columns if col != "is_churn"] + [
    "is_churn"
]
final_modeling_df = integrated_df[cols]


# 4. Saving the Prepared Dataset as CSV
final_csv_path = os.path.join(processed_dir, "final_modeling_dataset.csv")

print(f"Exporting final clean numeric dataset to CSV at:\n{final_csv_path}")

final_modeling_df.to_csv(final_csv_path, index=False)

print(f"Final Structured Shape for Modeling: {final_modeling_df.shape}")

display(final_modeling_df.head())

Handling missing values...



Encoding categorical column 'gender' into numeric flags...

Exporting final clean numeric dataset to CSV at:
C:\Users\workstation\Desktop\customer_churn_prediction\data\processed\final_modeling_dataset.csv
Final Structured Shape for Modeling: (992931, 29)


,msno,total_transactions,total_actual_paid,mean_actual_paid,mean_plan_days,total_cancellations,latest_auto_renew,latest_payment_method,average_under_overpayment,city,...,latest_date,total_active_days,skip_ratio,loop_ratio,secs_per_active_day,days_since_last_transaction,days_until_expiration,gender_male,gender_unknown,is_churn
0,waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,2,149,74.500000,18.500000,0,0,38,0.000000,18.0,...,2017-02-08,26.0,0.127863,23.017312,5823.713447,113,-83,0,0,1
1,QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,23,3458,150.347826,28.826087,2,1,39,6.478261,10.0,...,2017-03-19,512.0,0.151742,669.834658,5930.139491,65,-40,1,0,1
2,fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,10,1492,149.200000,30.000000,1,0,36,0.000000,11.0,...,2017-01-31,237.0,0.187306,202.341681,12452.014450,108,-86,0,0,1
3,mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,3,1937,645.666667,283.333333,0,0,17,0.000000,13.0,...,2017-03-31,725.0,0.324119,687.651492,8258.001404,34,-4,0,0,1
4,XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,9,4053,450.333333,97.222222,0,0,38,0.000000,3.0,...,2017-03-31,760.0,0.053332,700.840647,24347.028147,64,28,1,0,1


In [21]:


# 1. Drop the remaining date column that causes issues for Logistic Regression
if "latest_date" in integrated_df.columns:
    print("Dropping 'latest_date' column to ensure 100% numeric data...\n")
    integrated_df.drop(columns=["latest_date"], inplace=True)

# 2. Impute any missing values safely
print("Handling missing values...\n")
for col in integrated_df.select_dtypes(include=["number"]).columns:
    if col != "is_churn":
        integrated_df[col] = integrated_df[col].fillna(
            integrated_df[col].median()
        )

# Fix gender missing values before encoding
if "gender" in integrated_df.columns:
    print("Encoding categorical column 'gender' into numeric flags...\n")
    integrated_df["gender"] = integrated_df["gender"].fillna("unknown")
    integrated_df = pd.get_dummies(
        integrated_df, columns=["gender"], drop_first=True, dtype=int
    )

# 3. Column Reordering: Move the Target column (is_churn) to the very end
cols = [col for col in integrated_df.columns if col != "is_churn"] + [
    "is_churn"
]
final_modeling_df = integrated_df[cols]

# 4. Saving the Prepared Dataset as CSV
final_csv_path = os.path.join(processed_dir, "final_modeling_dataset.csv")
print("------------------------------------------------------------")
print(f"Exporting final clean numeric dataset to CSV at:\n{final_csv_path}")
print("------------------------------------------------------------\n")
final_modeling_df.to_csv(final_csv_path, index=False)

print(f"Final Structured Shape for Modeling: {final_modeling_df.shape}")

display(final_modeling_df.head())

Dropping 'latest_date' column to ensure 100% numeric data...

Handling missing values...

------------------------------------------------------------
Exporting final clean numeric dataset to CSV at:
C:\Users\workstation\Desktop\customer_churn_prediction\data\processed\final_modeling_dataset.csv
------------------------------------------------------------

Final Structured Shape for Modeling: (992931, 28)


,msno,total_transactions,total_actual_paid,mean_actual_paid,mean_plan_days,total_cancellations,latest_auto_renew,latest_payment_method,average_under_overpayment,city,...,total_secs,total_active_days,skip_ratio,loop_ratio,secs_per_active_day,days_since_last_transaction,days_until_expiration,gender_male,gender_unknown,is_churn
0,waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,2,149,74.500000,18.500000,0,0,38,0.000000,18.0,...,1.514166e+05,26.0,0.127863,23.017312,5823.713447,113,-83,0,0,1
1,QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,23,3458,150.347826,28.826087,2,1,39,6.478261,10.0,...,3.036231e+06,512.0,0.151742,669.834658,5930.139491,65,-40,1,0,1
2,fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,10,1492,149.200000,30.000000,1,0,36,0.000000,11.0,...,2.951128e+06,237.0,0.187306,202.341681,12452.014450,108,-86,0,0,1
3,mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,3,1937,645.666667,283.333333,0,0,17,0.000000,13.0,...,5.987051e+06,725.0,0.324119,687.651492,8258.001404,34,-4,0,0,1
4,XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,9,4053,450.333333,97.222222,0,0,38,0.000000,3.0,...,1.850374e+07,760.0,0.053332,700.840647,24347.028147,64,28,1,0,1
